In [6]:
import pandas as pd

# 1. Veriyi yükle (Dosya adının aynı klasörde olduğundan emin ol)
df = pd.read_csv('YAP471_Temiz_Kapanis_Verileri.csv')

# 2. Tarih sütununu indeks yap ki işlemler kolaylaşsın
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# 3. Günlük Getirileri (Daily Returns) hesapla
# pct_change() fonksiyonu (Bugün - Dün) / Dün işlemini otomatik yapar
returns = df.pct_change()

# 4. İlk satır boş kalacaktır (çünkü ilk günün bir öncesi yok), onu temizleyelim
returns = returns.dropna()

# Sonucu kontrol et
print("İlk 5 günlük getiri verisi:")
print(returns.head())

# İstersen bu sonuçları yeni bir dosya olarak kaydedebilirsin
# returns.to_csv('YAP471_Gunluk_Getiriler.csv')

İlk 5 günlük getiri verisi:
                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-03-04 -0.015773 -0.003683 -0.009131  0.011177 -0.048619 -0.033942
2021-03-05  0.010639  0.021487  0.007691  0.031058 -0.037799  0.007375
2021-03-08 -0.041650 -0.018156 -0.016164 -0.042718 -0.058450 -0.069641
2021-03-09  0.040622  0.028024  0.037561  0.016378  0.196412  0.080302
2021-03-10 -0.008997 -0.005784 -0.001698 -0.002046 -0.008195 -0.004163


In [7]:
# 5. 20 günlük hareketli (rolling) kovaryans matrisini hesapla
# Bu işlem, her tarih için hisseler arasındaki risk ilişkisini çıkarır
rolling_cov = returns.rolling(window=20).cov().dropna()

# Sonucu görmek için (Örneğin son tarihteki 6x6'lık matris)
print("\nEn güncel tarihteki Kovaryans Matrisi (Risk Tablosu):")
print(rolling_cov.tail(6))


En güncel tarihteki Kovaryans Matrisi (Risk Tablosu):
                      AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                        
2026-03-03 AAPL   0.000373  0.000047  0.000051 -0.000028  0.000055  0.000184
           MSFT   0.000047  0.000436  0.000109  0.000005  0.000191  0.000238
           AMZN   0.000051  0.000109  0.000417  0.000199  0.000047 -0.000084
           GOOGL -0.000028  0.000005  0.000199  0.000236  0.000027 -0.000010
           TSLA   0.000055  0.000191  0.000047  0.000027  0.000409  0.000382
           NVDA   0.000184  0.000238 -0.000084 -0.000010  0.000382  0.000871


In [8]:
# 1. Zeynep'ten gelen dosyayı oku
z_scores = pd.read_csv('YAP471_ZScore_Expected_Returns.csv')
z_scores['Date'] = pd.to_datetime(z_scores['Date'])
z_scores.set_index('Date', inplace=True)

# 2. Mean Reversion için sinyalleri ters çevir (Beklenen Getiri Vekili)
# Negatif Z-skoru -> Pozitif beklenen getiri demektir.
expected_returns = -z_scores

# 3. Tarihleri eşitle (Verilerin çakıştığı ortak günleri bul)
common_dates = returns.index.intersection(expected_returns.index)
expected_returns = expected_returns.loc[common_dates]

print("\nBeklenen Getiri Sinyalleri (İlk 5 Gün):")
print(expected_returns.head())


Beklenen Getiri Sinyalleri (İlk 5 Gün):
                AAPL      MSFT      AMZN     GOOGL      TSLA      NVDA
Date                                                                  
2021-03-30 -0.733271 -0.440287 -0.130543  0.056507 -0.427620  0.238385
2021-03-31  0.368465  0.604251  0.575656  0.584672  0.410937  1.296985
2021-04-01  0.712430  2.321058  1.780617  2.477413  0.197300  2.024444
2021-04-05  1.834357  2.975025  2.520373  3.351775  0.902504  2.016887
2021-04-06  1.901553  2.297084  2.283794  2.469782  0.934339  1.799743


In [9]:
import numpy as np
from scipy.optimize import minimize

def optimize_portfolio(expected_returns, cov_matrix):
    num_assets = len(expected_returns)
    args = (expected_returns, cov_matrix)

    # 1. Başlangıç tahmini (Eşit dağılım: 1/6)
    initial_guess = num_assets * [1. / num_assets]

    # 2. Kısıtlar: Ağırlıklar toplamı 1 olmalı
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

    # 3. Sınırlar: Her ağırlık 0 ile 1 arasında olmalı
    bounds = tuple((0, 1) for asset in range(num_assets))

    # 4. Amaç Fonksiyonu: Negatif Sharpe Oranı (veya Risk-Getiri Dengesi)
    # Burada basitleştirilmiş bir risk-getiri optimizasyonu yapıyoruz
    def objective(weights, returns, cov):
        portfolio_return = np.dot(weights, returns)
        portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov, weights)))
        # Amacımız getiriyi artırıp riski düşürmek (Minimize -Return/Risk)
        return -portfolio_return / portfolio_volatility

    result = minimize(objective, initial_guess, args=args,
                      method='SLSQP', bounds=bounds, constraints=constraints)

    return result.x

# Şimdi her tarih için bu optimizasyonu çalıştıralım
portfolio_weights = {}

for date in common_dates:
    try:
        # O güne ait getiri ve kovaryans verisini al
        daily_ret = expected_returns.loc[date].values
        daily_cov = rolling_cov.loc[date].values

        # Optimize et
        weights = optimize_portfolio(daily_ret, daily_cov)
        portfolio_weights[date] = weights
    except KeyError:
        continue

# Sonuçları DataFrame yapalım
weights_df = pd.DataFrame.from_dict(portfolio_weights, orient='index', columns=returns.columns)

print("\nOptimize Edilmiş Portföy Ağırlıkları (İlk 5 Gün):")
print(weights_df.head().round(4))


Optimize Edilmiş Portföy Ağırlıkları (İlk 5 Gün):
              AAPL    MSFT    AMZN   GOOGL  TSLA    NVDA
2021-03-31  0.0000  0.5155  0.0658  0.1736   0.0  0.2451
2021-04-01  0.0000  0.6606  0.0000  0.3394   0.0  0.0000
2021-04-05  0.0000  0.6235  0.0000  0.3765   0.0  0.0000
2021-04-06  0.0000  0.2422  0.3260  0.4317   0.0  0.0000
2021-04-07  0.1737  0.0000  0.8263  0.0000   0.0  0.0000


In [10]:
# Sonuçları CSV olarak kaydet
weights_df.to_csv('YAP471_Optimize_Portfolio_Weights.csv')
print("\nDosya 'YAP471_Optimize_Portfolio_Weights.csv' adıyla kaydedildi.")


Dosya 'YAP471_Optimize_Portfolio_Weights.csv' adıyla kaydedildi.
